In [1]:
# Optional: install required libraries for this lecture
%pip install -q openai


### Step 1: Load key (auth)
We load from environment. In client apps, never hardcode keys. Prefer a backend to keep keys secret.


In [2]:
import os, time
from getpass import getpass
from openai import OpenAI, RateLimitError, APITimeoutError

# Enter your OpenRouter API key when prompted.
OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key: ")

# OpenRouter provides an OpenAI-compatible API.
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

# You can change this to another OpenRouter-supported model.
MODEL = "openai/gpt-4o-mini"

client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_BASE_URL,
    timeout=10,
)

print("OpenRouter key loaded:", bool(OPENROUTER_API_KEY))
print("Model:", MODEL)


Enter your OpenRouter API key: ··········
OpenRouter key loaded: True
Model: openai/gpt-4o-mini


### Step 2: Direct client call (frontend)
Fastest to prototype, but exposes model choices and risks key leakage if done purely client-side. Safer in server-only apps.


In [3]:
def direct_call(prompt: str) -> str:
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    return resp.choices[0].message.content


### Step 3: Backend proxy pattern (recommended for apps)
Hide the OpenAI key on the server. Frontend calls your `/api/chat` endpoint; the server forwards to OpenAI, applies retries, logs usage, and returns only what you need.


In [4]:
import requests

# Example of how a frontend would call your backend (simulate with a placeholder URL)
# In real apps, implement /api/chat on your server with the OpenRouter key and logic below.

def backend_proxy_call(prompt: str) -> str:
    # Simulate a backend: we just call OpenRouter here for demo, but in production this lives on the server.
    return direct_call(prompt)

print("Direct:", direct_call("Say hello in 6 words."))
print("Backend proxy:", backend_proxy_call("Say hello in 6 words."))


Direct: Hello! How are you doing today?
Backend proxy: Hello! How are you doing today?


### Step 4: Auth strategies (secure by default)
- Do not ship `OPENAI_API_KEY` to the browser. Keep it server-side.
- Use a backend proxy endpoint (e.g., `/api/chat`) to attach the key.
- Rotate keys and set sensible quotas. Never print secrets in logs.


### Step 5: Reliability — simple retry + backoff
Handle rate limits/timeouts gracefully. Keep retries small (e.g., 2–3) and add jitter in real systems.


In [5]:
def call_with_retry(prompt: str, retries: int = 2) -> str:
    delay = 1
    for attempt in range(retries + 1):
        try:
            return direct_call(prompt)
        except (RateLimitError, APITimeoutError):
            if attempt == retries:
                raise
            time.sleep(delay)
            delay *= 2

print("Retry demo:", call_with_retry("Say a 5-word greeting."))


Retry demo: Hello! Hope you're having a great day!


### Step 6: Separation of concerns (tiny functions)
- `build_messages(...)` for prompt design
- `call_openai(...)` for API
- `postprocess(...)` for output handling
Small, readable pieces you can test and swap.


In [6]:
def build_messages(context: str, question: str):
    return [
        {"role": "system", "content": "Answer using ONLY the provided context. If missing, say 'I don't know.'"},
        {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
    ]

def call_openai(messages):
    resp = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0,
    )
    return resp.choices[0].message.content

def postprocess(text: str) -> str:
    return text.strip()

msg = build_messages("Module 3 teaches OpenAI, Gemini, Anthropic.", "Which providers are included?")
print("SoC demo:", postprocess(call_openai(msg)))


SoC demo: OpenAI, Gemini, Anthropic.
